# 📈 Tek Yönlü ANOVA Testi ve Post-Hoc Analizleri (Analysis of Variance)

ANOVA (Varyans Analizi), **üç veya daha fazla bağımsız grubun** ortalamaları arasında istatistiksel olarak anlamlı bir fark olup olmadığını test etmek için kullanılır.

---

## 🎯 1. Neden Çoklu T-Testi Yerine ANOVA?

3 grubumuz olduğunda ($A, B, C$), ikili t-testleri yaparsak ($A-B, A-C, B-C$) toplam 3 test gerekir.
Her testte $\alpha = 0.05$ kabul edilirse, **I. Tip Hata Oranı (Family-wise Error Rate)** katlanarak artar:
$$\alpha_{\text{toplam}} = 1 - (1 - 0.05)^3 \approx 14.3\%$$

ANOVA bu problemi çözer ve tüm grupları **tek bir F-istatistiği** ile aynı anda test eder.

### 📐 ANOVA Hipotezleri:
- **$H_0$ (Sıfır Hipotezi):** $\mu_1 = \mu_2 = \mu_3 = \dots = \mu_k$ *(Tüm grupların ortalamaları eşittir)*
- **$H_1$ (Alternatif Hipotez):** En az bir grup ortalaması diğerlerinden farklıdır.

### 🧮 F-İstatistiği Formülü:
$$F = \frac{\text{Gruplar Arası Varyans (Between-Group Variance)}}{\text{Grup İçi Varyans (Within-Group Variance)}} = \frac{MS_{\text{between}}}{MS_{\text{within}}}$$


In [ ]:
import pandas as pd
import scipy.stats as stats

# 3 Farklı Öğrenme Metodu (A, B, C) Skorları
A = [78, 85, 82, 88, 90]
B = [72, 75, 78, 74, 77]
C = [90, 92, 94, 89, 91]

f_stat, p_value = stats.f_oneway(A, B, C)

print(f"F-İstatistiği: {f_stat:.4f}")
print(f"p-değeri     : {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(" Sonuç: H0 reddedilir! Metotlar arasında istatistiksel olarak anlamlı bir fark vardır.")
else:
    print(" Sonuç: H0 kabul edilir. Metotlar arasında anlamlı bir fark tespit edilememiştir.")


---
## 📊 2. Pandas DataFrame ile One-Way ANOVA Uygulaması

Gerçek projelerde veriler genellikle kategorik grup sütunları ve nümerik değer sütunları şeklinde saklanır.


In [ ]:
import pandas as pd
from scipy.stats import f_oneway

df = pd.DataFrame({
    "Grup": ["A", "A", "B", "B", "C", "C"],
    "Skor": [80, 82, 78, 76, 90, 88]
})

print(df)

# Grupları ayırma
A = df[df["Grup"] == "A"]["Skor"]
B = df[df["Grup"] == "B"]["Skor"]
C = df[df["Grup"] == "C"]["Skor"]

f_stat, p_value = f_oneway(A, B, C)
print(f"
F-İstatistiği: {f_stat:.4f}, p-değeri: {p_value:.6f}")

alpha = 0.05
if p_value < alpha:
    print(" Sonuç: H0 reddedilir. Grup skorları arasında anlamlı fark vardır.")
else:
    print(" Sonuç: H0 reddedilemez.")


---
## 🔍 3. Post-Hoc Analiz: Tukey HSD (Honestly Significant Difference)

> **Soru:** ANOVA $H_0$ hipotezini reddettiğinde *"Fark var"* der; fakat **farkın hangi gruplar arasında olduğunu söylemez**.
> **Çözüm:** Çoklu karşılaştırma düzeltmesi yapan **Tukey HSD testi** ile hangi ikililerin birbirinden anlamlı düzeyde ayrıştığını buluruz.


In [ ]:
import pandas as pd
from statsmodels.stats.multicomp import pairwise_tukeyhsd
import matplotlib.pyplot as plt

data = pd.DataFrame({
    "Kafe" : ["StarCafe"]*4 + ["JavaHouse"]*4 + ["BookBucks"]*4,
    "Zaman": ["Sabah", "Sabah", "Akşam", "Akşam"] * 3,
    "Verimlilik": [25, 26, 22, 21,
                   19, 20, 21, 22,
                   30, 31, 28, 27]
})

data["Grup"] = data["Kafe"] + "-" + data["Zaman"]

# Tukey HSD Testi
tukey = pairwise_tukeyhsd(endog=data["Verimlilik"], groups=data["Grup"], alpha=0.05)
print(tukey)

# Güven Aralığı Görselleştirmesi
fig, ax = plt.subplots(figsize=(10, 6))
tukey.plot_simultaneous(ax=ax)
plt.title("Tukey HSD - Grup İkili Karşılaştırmaları (%95 Güven Aralığı)")
plt.xlabel("Verimlilik Puanı")
plt.show()


---
## 🛒 4. Satış Yöntemlerinin Karşılaştırılması & Seaborn Görselleştirme

3 farklı satış stratejisinin ($A, B, C$) satış rakamları üzerindeki etkisi incelenmektedir.


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import f_oneway

df_satis = pd.DataFrame({
    "Yontem": ["A"] * 5 + ["B"] * 5 + ["C"] * 5,
    "Satis": [100, 105, 98, 102, 101, 92, 88, 95, 90, 93, 110, 115, 112, 117, 113]
})

plt.figure(figsize=(8, 5))
sns.barplot(x="Yontem", y="Satis", data=df_satis, palette="Blues_d", capsize=0.1)
plt.title("Satış Yöntemlerine Göre Ortalama Satış Miktarları", fontsize=13)
plt.ylabel("Satış Adedi")
plt.xlabel("Satış Yöntemi")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.show()

A = df_satis[df_satis["Yontem"] == "A"]["Satis"]
B = df_satis[df_satis["Yontem"] == "B"]["Satis"]
C = df_satis[df_satis["Yontem"] == "C"]["Satis"]

f_stat, p_value = f_oneway(A, B, C)
print(f"F-İstatistiği: {f_stat:.4f}, p-değeri: {p_value:.8f}")

if p_value < 0.05:
    print(" Sonuç: H0 reddedilir! Satış yöntemleri arasında istatistiksel olarak çok güçlü bir fark vardır.")
else:
    print(" Sonuç: H0 kabul edilir. Fark önemsizdir.")
